### Task 2 — Clean the Data & Save as CSV

This section will handle loading the raw JSON data, cleaning it according to the specified criteria, and then saving the processed data into a tidy CSV file.

### 2 — Clean the Data

This step involves addressing common data quality issues such as duplicates, missing values, incorrect data types, and inconsistent formatting.

### 3 — Save as CSV

Finally, the cleaned data will be saved to a CSV file and a summary will be provided.

In [1]:
# Import the pandas library for data manipulation
import pandas as pd
import os
import json

# Define the path to the JSON file from Task 1
json_file_path = 'data/trends_20260906.json'

# Ensure the 'data' directory exists
if not os.path.exists('data'):
    os.makedirs('data')

# Create a dummy JSON file for demonstration if it doesn't exist
if not os.path.exists(json_file_path):
    dummy_data = [
        {"post_id": "1", "title": "  Amazing Tech  ", "score": 10, "num_comments": 5, "category": "technology"},
        {"post_id": "2", "title": "World News Update", "score": 6, "num_comments": 10, "category": "worldnews"},
        {"post_id": "1", "title": "  Amazing Tech  ", "score": 10, "num_comments": 5, "category": "technology"}, # Duplicate
        {"post_id": "3", "title": "Sports Highlight", "score": 4, "num_comments": 2, "category": "sports"}, # Low quality
        {"post_id": "4", "title": "Science Breakthrough", "score": 15, "num_comments": 20, "category": "science"},
        {"post_id": "5", "title": None, "score": 8, "num_comments": 3, "category": "entertainment"}, # Missing title
        {"post_id": "6", "title": "Another Tech Story", "score": 7, "num_comments": 12, "category": "technology"},
        {"post_id": None, "title": "Missing ID", "score": 9, "num_comments": 4, "category": "worldnews"}, # Missing post_id
        {"post_id": "7", "title": "Entertainment Tonight", "score": 12, "num_comments": 7, "category": "entertainment"}
    ]
    with open(json_file_path, 'w') as f:
        json.dump(dummy_data, f, indent=4)
    print(f"Dummy JSON file created at {json_file_path}")

# 1 — Load the JSON File
try:
    df = pd.read_json(json_file_path)
    print(f"Loaded {len(df)} stories from {json_file_path}")
except FileNotFoundError:
    print(f"Error: The file {json_file_path} was not found. Please ensure it exists.")
    df = pd.DataFrame() # Create an empty DataFrame to avoid errors in subsequent steps

# Make a copy to work on, preserving the original loaded data
df_cleaned = df.copy()

# --- Duplicates — remove any rows with the same post_id ---
initial_rows = len(df_cleaned)
df_cleaned.drop_duplicates(subset=['post_id'], inplace=True)
print(f"After removing duplicates: {len(df_cleaned)}")

# --- Missing values — drop rows where post_id, title, or score is missing ---
rows_before_null_check = len(df_cleaned)
df_cleaned.dropna(subset=['post_id', 'title', 'score'], inplace=True)
print(f"After removing nulls: {len(df_cleaned)}")

# --- Data types — make sure score and num_comments are integers ---
# Convert 'score' to integer, coercing errors to NaN and then dropping rows with NaN
df_cleaned['score'] = pd.to_numeric(df_cleaned['score'], errors='coerce')
df_cleaned.dropna(subset=['score'], inplace=True) # Drop rows where score became NaN after coercion
df_cleaned['score'] = df_cleaned['score'].astype(int)

# Convert 'num_comments' to integer, coercing errors to NaN and then dropping rows with NaN
df_cleaned['num_comments'] = pd.to_numeric(df_cleaned['num_comments'], errors='coerce')
df_cleaned.dropna(subset=['num_comments'], inplace=True) # Drop rows where num_comments became NaN after coercion
df_cleaned['num_comments'] = df_cleaned['num_comments'].astype(int)

# --- Low quality — remove stories where score is less than 5 ---
rows_before_low_score_check = len(df_cleaned)
df_cleaned = df_cleaned[df_cleaned['score'] >= 5]
print(f"After removing low scores: {len(df_cleaned)}")

# --- Whitespace — strip extra spaces from the title column ---
df_cleaned['title'] = df_cleaned['title'].str.strip()

# Print the final number of rows remaining after cleaning
print(f"Total rows remaining after cleaning: {len(df_cleaned)}")

# Define the path for the cleaned CSV file
csv_file_path = 'data/trends_clean.csv'

# Save the cleaned DataFrame to a CSV file
df_cleaned.to_csv(csv_file_path, index=False)

# Print a confirmation message with the number of rows saved
print(f"Saved {len(df_cleaned)} rows to {csv_file_path}")

# Print a quick summary: how many stories per category
print("\nStories per category:")
print(df_cleaned['category'].value_counts().to_string())

Dummy JSON file created at data/trends_20260906.json
Loaded 9 stories from data/trends_20260906.json
After removing duplicates: 8
After removing nulls: 6
After removing low scores: 5
Total rows remaining after cleaning: 5
Saved 5 rows to data/trends_clean.csv

Stories per category:
category
technology       2
worldnews        1
science          1
entertainment    1
